# TP Final: SPAM Classification
## Machine Learning - Use Case

**Objectif**: Classifier un SMS comme SPAM ou HAM (pas spam) en utilisant du Machine Learning

**Dataset**: SMS Spam Collection - 5572 SMS étiquetés

**Technologies**: pandas, scikit-learn, spaCy, regex

## Étape 0: Installation des packages

In [ ]:
# Importer les libraries
import pandas as pd
import numpy as np
import re
import spacy
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns


## Étape 1: Télécharger et charger les données

In [ ]:
# Télécharger les données
!wget https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip -O spam_data.zip

In [ ]:
# Extraire le ZIP
import zipfile
with zipfile.ZipFile('spam_data.zip', 'r') as zip_ref:
    zip_ref.extractall()

print("Fichiers extraits!")

In [ ]:
# Charger les données dans un DataFrame
# Format: label \t message
df = pd.read_csv('SMSSpamCollection', sep='\t', names=['label', 'message'])

print(f"Données chargées: {len(df)} SMS")
print(f"\nPremières lignes:")
print(df.head())

## Étape 2: Exploration Data Analysis (1) - Données brutes

In [ ]:
# Analyser les données manquantes
print("Données manquantes:")
print(df.isnull().sum())
print(f"\nPas de données manquantes!")

In [ ]:
# Distribution des classes
print("Distribution des classes:")
print(df['label'].value_counts())
print(f"\nProportions:")
print(df['label'].value_counts(normalize=True) * 100)

# Visualiser
plt.figure(figsize=(8, 5))
df['label'].value_counts().plot(kind='bar', color=['green', 'red'])
plt.title('Distribution SPAM vs HAM')
plt.xlabel('Classe')
plt.ylabel('Nombre de SMS')
plt.show()

In [ ]:
# Regarder quelques exemples
print("Exemples de HAM:")
print(df[df['label'] == 'ham']['message'].head(3).values)

print("\nExemples de SPAM:")
print(df[df['label'] == 'spam']['message'].head(3).values)

## Étape 3: Preprocessing du texte

In [ ]:
# Charger le modèle spaCy français
nlp = spacy.load('fr_core_news_sm')

# Fonction de preprocessing
def preprocessing(text):
    """
    Preprocessing complet d'un texte:
    1. Remplacer les caractères non-alphabétiques par un espace
    2. Convertir en minuscules
    3. Tokenization avec spaCy
    4. Supprimer les stop words
    5. Supprimer les tokens de moins de 3 caractères
    6. Lemmatization
    7. Reconstruire la phrase
    """
    # 1. Remplacer nombres et caractères spéciaux
    text = re.sub(r"[^a-zA-Z]", " ", str(text))
    
    # 2. Minuscules
    text = text.lower()
    
    # 3. spaCy processing
    doc = nlp(text)
    
    # 4-6. Stop words, longueur, lemmatization
    tokens = [
        token.lemma_ for token in doc
        if not token.is_stop and len(token.text) >= 3
    ]
    
    # 7. Reconstruire
    return " ".join(tokens)

# Tester sur le premier SMS
test_sms = df.iloc[0]['message']
print(f"Original: {test_sms}")
print(f"\nPreprocessed: {preprocessing(test_sms)}")

## Étape 4: Appliquer le preprocessing à tous les SMS

In [ ]:
# Attention: Cela peut prendre quelques minutes...
# Appliquer preprocessing à tous les messages
print("Preprocessing en cours... (cela peut prendre quelques minutes)")
df['message_clean'] = df['message'].apply(preprocessing)
print("Preprocessing terminé!")

# Sauvegarder les résultats
df.to_csv('spam_dataset_cleaned.csv', index=False)
print("Données sauvegardées dans 'spam_dataset_cleaned.csv'")

## 🔍 Étape 5: EDA (2) - Données prétraitées

In [ ]:
# Regarder les résultats du preprocessing
print("Exemples après preprocessing:")
for i in range(3):
    sms = df.iloc[i]
    print(f"\n[{sms['label'].upper()}]")
    print(f"  Original:     {sms['message'][:80]}...")
    print(f"  Cleaned:      {sms['message_clean'][:80]}...")

In [ ]:
# Analyser la longueur des messages
df['message_length'] = df['message_clean'].apply(len)
df['word_count'] = df['message_clean'].apply(lambda x: len(x.split()))

print(" Statistiques sur les messages:")
print(df.groupby('label')[['message_length', 'word_count']].describe())

In [ ]:
# Charger le modèle spaCy anglais (le dataset est en anglais!)\nnlp = spacy.load('en_core_web_sm')\n\ndef preprocessing(text):\n    \"\"\"\n    Preprocessing complet d'un texte:\n    1. Remplacer les caractères non-alphabétiques par un espace\n    2. Convertir en minuscules\n    3. Tokenization avec spaCy\n    4. Supprimer les stop words\n    5. Supprimer les tokens de moins de 3 caractères\n    6. Lemmatization\n    7. Reconstruire la phrase\n    \"\"\"\n    # 1. Remplacer nombres et caractères spéciaux\n    text = re.sub(r\"[^a-zA-Z]\", \" \", str(text))\n    \n    # 2. Minuscules\n    text = text.lower()\n    \n    # 3. spaCy processing\n    doc = nlp(text)\n    \n    # 4-6. Stop words, longueur, lemmatization\n    tokens = [\n        token.lemma_ for token in doc\n        if not token.is_stop and len(token.text) >= 3\n    ]\n    \n    # 7. Reconstruire\n    return \" \".join(tokens)

## Étape 6: Machine Learning

In [ ]:
# Préparer les données X (features) et y (labels)
X = df['message_clean']
y = df['label'].map({'ham': 0, 'spam': 1})  # 0=ham, 1=spam

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nDonnées prêtes pour le ML!")

In [ ]:
# Train/Test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {len(X_train)} SMS")
print(f"Test set:  {len(X_test)} SMS")
print(f"\nDistribution train: {y_train.value_counts().to_dict()}")
print(f"Distribution test:  {y_test.value_counts().to_dict()}")

In [ ]:
# Vectorization avec Bag of Words (CountVectorizer)
print("Vectorization Bag of Words...")
vectorizer_bow = CountVectorizer(max_features=5000)
X_train_bow = vectorizer_bow.fit_transform(X_train)
X_test_bow = vectorizer_bow.transform(X_test)

print(f" BoW shape: {X_train_bow.shape}")
print(f"   (nombre de SMS x nombre de features)")

In [ ]:
# Vectorization avec TF-IDF
print("Vectorization TF-IDF...")
vectorizer_tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer_tfidf.fit_transform(X_train)
X_test_tfidf = vectorizer_tfidf.transform(X_test)

print(f" TF-IDF shape: {X_train_tfidf.shape}")

In [ ]:
# Entraîner Naive Bayes avec Bag of Words
print("Entraînement Naive Bayes (BoW)...")
nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)
print("Modèle Naive Bayes (BoW) entraîné!")

# Prédiction
y_pred_bow = nb_bow.predict(X_test_bow)
accuracy_bow = accuracy_score(y_test, y_pred_bow)
print(f"\nAccuracy (BoW): {accuracy_bow:.2%}")

In [ ]:
# Entraîner Naive Bayes avec TF-IDF
print("Entraînement Naive Bayes (TF-IDF)...")
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)
print("Modèle Naive Bayes (TF-IDF) entraîné!")

# Prédiction
y_pred_tfidf = nb_tfidf.predict(X_test_tfidf)
accuracy_tfidf = accuracy_score(y_test, y_pred_tfidf)
print(f"\nAccuracy (TF-IDF): {accuracy_tfidf:.2%}")

## Étape 7: Évaluation détaillée

In [ ]:
# Choisir le meilleur modèle (habituellement TF-IDF est meilleur)
if accuracy_tfidf > accuracy_bow:
    best_model = nb_tfidf
    best_vectorizer = vectorizer_tfidf
    y_pred = y_pred_tfidf
    best_name = "TF-IDF"
else:
    best_model = nb_bow
    best_vectorizer = vectorizer_bow
    y_pred = y_pred_bow
    best_name = "Bag of Words"

print(f"Meilleur modèle: Naive Bayes ({best_name})")

In [ ]:
# Confusion Matrix
print("\n MATRICE DE CONFUSION:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Visualiser
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['HAM', 'SPAM'],
            yticklabels=['HAM', 'SPAM'])
plt.title(f'Confusion Matrix - {best_name}')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
# Metrics détaillées
print("\n MÉTRIQUES D'ÉVALUATION:")
print(f"\nAccuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.4f}")

print("\n Interprétation:")
print(f"  - Accuracy: {accuracy_score(y_test, y_pred):.1%} des prédictions correctes")
print(f"  - Precision: {precision_score(y_test, y_pred):.1%} des SPAM détectés sont vrais")
print(f"  - Recall: {recall_score(y_test, y_pred):.1%} des SPAM réels sont détectés")
print(f"  - F1-Score: Balance entre Precision et Recall")

## Étape 8: Tester sur de nouvelles phrases

In [ ]:
def predict_spam(sentence):
    """
    Prédire si une nouvelle phrase est du SPAM ou HAM
    """
    # Preprocessing
    cleaned = preprocessing(sentence)
    
    # Vectorization
    vectorized = best_vectorizer.transform([cleaned])
    
    # Prédiction
    prediction = best_model.predict(vectorized)[0]
    probability = best_model.predict_proba(vectorized)[0]
    
    label = "SPAM" if prediction == 1 else "HAM"
    
    print(f"Phrase: {sentence}")
    print(f"Prédiction: {label}")
    print(f"Confiance SPAM: {probability[1]:.1%}")
    print(f"Confiance HAM:  {probability[0]:.1%}")
    print()
    
    return prediction

# Tester avec des phrases différentes
print("TESTS SUR DE NOUVELLES PHRASES:\n")

test_sentences = [
    "send a free txt with your mobile",
    "Hi, how are you doing today?",
    "CONGRATULATIONS! You won a FREE prize! Click here NOW!",
    "Let's meet tomorrow at 5pm",
    "URGENT: Confirm your bank details to claim your inheritance!",
    "What time is the meeting?",
]

for sentence in test_sentences:
    predict_spam(sentence)

## Résumé du projet

### Pipeline complet:
1. ✅ **Data Collection** - SMS Spam Collection (5572 messages)
2. ✅ **EDA** - Analyse initiale et distribution
3. ✅ **Preprocessing** - Cleaning avec regex et spaCy
4. ✅ **Vectorization** - BoW et TF-IDF
5. ✅ **Model Training** - Naive Bayes
6. ✅ **Evaluation** - Accuracy, Precision, Recall, F1
7. ✅ **Deployment** - Prédiction sur nouvelles phrases

### Points clés:
- **Naive Bayes** est une bonne baseline pour la classification de texte
- **TF-IDF** généralement meilleur que Bag of Words
- **Preprocessing** crucial pour de bonnes performances
- **F1-Score** mieux qu'Accuracy pour données déséquilibrées
- **Confiance de prédiction** fournie par `predict_proba()`